# Wanderbricks Data Ingestion to Bronze Layer

## Overview
This notebook ingests all tables from the `samples.wanderbricks` schema into the bronze layer.

## Configuration
* **Widget**: `catalog_name` (default: "workspace") - The target catalog for bronze tables
* **Target Schema**: `{catalog_name}.bronze_raw`
* **Source Schema**: `samples.wanderbricks`

## Tables Ingested
This notebook ingests the following 16 tables:
1. `amenities` → `wanderbricks_amenities`
2. `booking_updates` → `wanderbricks_booking_updates`
3. `bookings` → `wanderbricks_bookings`
4. `clickstream` → `wanderbricks_clickstream`
5. `countries` → `wanderbricks_countries`
6. `customer_support_logs` → `wanderbricks_customer_support_logs`
7. `destinations` → `wanderbricks_destinations`
8. `employees` → `wanderbricks_employees`
9. `hosts` → `wanderbricks_hosts`
10. `page_views` → `wanderbricks_page_views`
11. `payments` → `wanderbricks_payments`
12. `properties` → `wanderbricks_properties`
13. `property_amenities` → `wanderbricks_property_amenities`
14. `property_images` → `wanderbricks_property_images`
15. `reviews` → `wanderbricks_reviews`
16. `users` → `wanderbricks_users`

## Process
Each table is:
1. Read from the source using `spark.table()`
2. Row count is captured before writing
3. Written to the bronze layer with `overwrite` mode
4. Prefixed with `wanderbricks_` to identify the source schema

In [0]:
# Create widget for catalog name
dbutils.widgets.text("catalog_name", "workspace", "Catalog Name")
catalog_name = dbutils.widgets.get("catalog_name")

In [0]:
# Create bronze_raw schema if it doesn't exist
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.bronze_raw")

In [0]:
from common.table_utils import get_source_table, get_target_table

def ingest_table(table_name):
    """
    Ingest a table from samples.wanderbricks into the bronze layer.
    
    Args:
        table_name: Name of the table in samples.wanderbricks schema
    
    Returns:
        int: Number of rows ingested
    """
    source_table = get_source_table("wanderbricks", table_name)
    target_table = get_target_table(catalog_name, "bronze_raw", "wanderbricks", table_name)
    
    # Read from source using SQL
    df = spark.sql(f"SELECT * FROM {source_table}")
    row_count = df.count()
    
    # Write to target
    df.write.mode("overwrite").saveAsTable(target_table)
    
    return row_count

In [0]:
# Array of table names to ingest from samples.wanderbricks
table_names = [
    "amenities",
    "booking_updates",
    "bookings",
    "clickstream",
    "countries",
    "customer_support_logs",
    "destinations",
    "employees",
    "hosts",
    "page_views",
    "payments",
    "properties",
    "property_amenities",
    "property_images",
    "reviews",
    "users"
]

In [0]:
# Loop through each table and ingest
for table_name in table_names:
    row_count = ingest_table(table_name)
    print(f"Ingested {table_name}: {row_count} rows")

print(f"\nCompleted ingestion of {len(table_names)} tables from samples.wanderbricks to {catalog_name}.bronze_raw")

In [0]:
%sql
SHOW TABLES IN IDENTIFIER(:catalog_name).bronze_raw